# 03 - NewsQA mini dataset + evidence-to-chunk mapping

Downloads ~10-20 NewsQA articles and builds a small evaluation dataset. Two goals:
1. **Article evaluation dataset** - select N articles + their questions.
2. **Evidence-to-chunk mapping** - turn each answer's evidence into `relevant_chunk_ids`.

Output: `data/testset_mini.jsonl` (schema = `docs/evaluation.md` section 6.1), ready for
`scripts/run_benchmark.py` and the dashboard.

### Do we need to crawl the source articles? **No.**
`lucadiliello/newsqa` ships each article's **full body inline** as `context`, and the evidence as
**character offsets** (`labels.start/end`) into that context. So both goals are done fully offline -
no title/URL crawl needed. NewsQA has no clean title/URL/chunk_id, so we derive:
- `article_id = md5(context)[:12]` (matches `chunker.generate_article_id`, so chunk IDs are reproducible),
- a readable title from the article's first line.

The live crawler (`crawler/`) fetches *today's* news; it can't recover a 2016 CNN URL from body text,
so it adds nothing here. Use it only if you later want real title/URL metadata.

## 0. Setup

In [ ]:
import os, sys, json, hashlib, subprocess
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
    os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import yaml
config = yaml.safe_load(open('configs/config.yaml', encoding='utf-8'))
from src.ingestion.chunker import get_chunker
chunker = get_chunker(config)

N_ARTICLES = 15                      # target articles (goal: 10-20)
MAX_SCAN   = 800                     # NewsQA rows to stream (multiple Qs share an article)
OUT        = 'data/testset_mini.jsonl'
print('chunk config:', config['chunking'])

## 1. Download + group by article  (goal 1)
Stream NewsQA and group rows by `context`; each article carries all its questions.

In [ ]:
from datasets import load_dataset

ds = load_dataset('lucadiliello/newsqa', split='train', streaming=True)
groups = {}                          # context -> [row, ...], insertion-ordered
for i, s in enumerate(ds):
    if i >= MAX_SCAN:
        break
    groups.setdefault(s['context'], []).append(s)

picked = list(groups.items())[:N_ARTICLES]
print(f'scanned {MAX_SCAN} rows -> {len(groups)} articles; picked {len(picked)}')

import pandas as pd
pd.DataFrame([
    {'article_id': 'newsqa_' + hashlib.md5(ctx.encode()).hexdigest()[:12],
     'title': ctx.splitlines()[0][:60], 'n_questions': len(rows), 'chars': len(ctx)}
    for ctx, rows in picked
])

## 2. Inspect one article (raw structure + evidence highlighted)
See exactly what a NewsQA row gives us before any processing.

In [ ]:
def evidence_span(sample, ctx):
    """(start, end, text) of the answer evidence in ctx via char offsets; fallback to answer substring."""
    lab = sample.get('labels') or []
    if lab and lab[0].get('start') and lab[0].get('end'):
        a, b = int(lab[0]['start'][0]), int(lab[0]['end'][0]) + 1   # end is inclusive in NewsQA
        if 0 <= a < b <= len(ctx):
            return a, b, ctx[a:b]
    ans = (sample.get('answers') or [''])[0]
    i = ctx.find(ans)
    return (i, i + len(ans), ans) if i >= 0 else (None, None, ans)

ctx0, rows0 = picked[0]
print('ARTICLE:', ctx0.splitlines()[0][:80], chr(10))
for s in rows0[:5]:
    a, b, ev = evidence_span(s, ctx0)
    around = ctx0[max(0, a-40):a] + '<<' + ev + '>>' + ctx0[b:b+40] if a is not None else '(no span)'
    print('Q:', s['question'])
    print('  answer  :', s['answers'])
    print('  evidence:', repr(ev), f'@ [{a}:{b}]')
    print('  context :', around.replace(chr(10), ' '), chr(10))

## 3. Chunk the article (project chunker, production config)
Same chunker/config that builds the live collection, so chunk IDs match end to end.

In [ ]:
article_key0 = 'newsqa_' + hashlib.md5(ctx0.encode()).hexdigest()[:12]
chunks0 = chunker.chunk_article(
    {'text': ctx0, 'metadata': {'url': '', 'title': ctx0.splitlines()[0][:80], 'publisher': 'CNN'}},
    filename=article_key0,
)
for c in chunks0:
    print(c['id'], '|', len(c['text']), 'chars |', c['text'][:70].replace(chr(10), ' '), '...')

## 4. Evidence -> chunk mapping  (goal 2, the core)
Locate each chunk's true `[start, end)` in the article, then a question's evidence maps to every chunk
whose range overlaps `[evidence_start, evidence_end)`. Exact and offset-based; falls back to a
substring match only if offsets drift (whitespace). This is the logic to extend in `src/evaluation/testset.py`.

In [ ]:
def chunk_ranges(ctx, chunks):
    """True [start, end) of each chunk within ctx, walking forward (handles overlap)."""
    ranges, cur = [], 0
    for c in chunks:
        pos = ctx.find(c['text'], cur)
        if pos < 0:                       # whitespace drift: retry on a short prefix
            pos = ctx.find(c['text'][:40])
        if pos < 0:
            ranges.append((None, None)); continue
        ranges.append((pos, pos + len(c['text'])))
        cur = pos + 1                     # next chunk may overlap this one, so +1 not +len
    return ranges

def map_evidence(chunks, ranges, a, b, ev):
    """Chunk IDs whose char-range overlaps the evidence span; substring fallback."""
    if a is not None:
        hit = [c['id'] for c, (s, e) in zip(chunks, ranges)
               if s is not None and s < b and a < e]
        if hit:
            return hit
    evl = ev.lower()
    return [c['id'] for c in chunks if evl and evl in c['text'].lower()]

ranges0 = chunk_ranges(ctx0, chunks0)
print('chunk ranges:', ranges0, chr(10))
for s in rows0[:5]:
    a, b, ev = evidence_span(s, ctx0)
    print(map_evidence(chunks0, ranges0, a, b, ev), '<-', s['question'])

## 5. Build the mini dataset -> `data/testset_mini.jsonl`  (goals 1+2)
Runs steps 3-4 over every article and writes the JSONL (schema `docs/evaluation.md` 6.1).

In [ ]:
entries, all_chunks = [], []
for ctx, rows in picked:
    akey = 'newsqa_' + hashlib.md5(ctx.encode()).hexdigest()[:12]
    chunks = chunker.chunk_article(
        {'text': ctx, 'metadata': {'url': '', 'title': ctx.splitlines()[0][:80], 'publisher': 'CNN'}},
        filename=akey)
    ranges = chunk_ranges(ctx, chunks)
    all_chunks.extend(chunks)
    for s in rows:
        a, b, ev = evidence_span(s, ctx)
        entries.append({
            'question': s['question'],
            'ground_truth': (s.get('answers') or [''])[0],
            'article_key': akey,
            'relevant_chunk_ids': map_evidence(chunks, ranges, a, b, ev),
            'evidence': ev,
            'article_chunk_ids': [c['id'] for c in chunks],
        })

mapped = sum(1 for e in entries if e['relevant_chunk_ids'])
assert mapped == len(entries), f'{len(entries)-mapped} questions failed to map - inspect them'
Path(OUT).parent.mkdir(parents=True, exist_ok=True)
with open(OUT, 'w', encoding='utf-8') as f:
    for e in entries:
        f.write(json.dumps(e, ensure_ascii=False) + chr(10))
print(f'wrote {OUT}: {len(picked)} articles, {len(entries)} questions, mapped {mapped}/{len(entries)}')

## 6. (Optional) Verify end to end
Build a throwaway Chroma collection from these same articles and benchmark the mini set - proves the
chunk IDs line up. Retrieval-only, **no API cost**. NewsQA articles are short (1-2 chunks each), so
scores will look easy; the point here is that the mapping resolves, not the difficulty.

In [ ]:
from src.indexing.embeddings import get_embedding_function
from src.indexing.chroma_store import ChromaStore

DB, MINI = 'data/chroma_db', 'newsqa_mini'
store = ChromaStore(DB, get_embedding_function(config))
store.get_or_create_collection(MINI, hnsw_config=config['database']['hnsw'])
print('upserted:', store.upsert_chunks(MINI, all_chunks))

cmd = [sys.executable, 'scripts/run_benchmark.py', '--retriever', 'dense',
       '--testset', OUT, '--db-path', DB, '--collection', MINI,
       '--report-dir', 'reports/mini']
subprocess.run(cmd, check=True)
# Clean up the throwaway collection when done:
# store.delete_collection(MINI)

## Done
`data/testset_mini.jsonl` is a valid test set. Point notebook 02 (or `run_benchmark.py`) at it with
`--collection newsqa_mini` to score, and the dashboard picks up `reports/mini`. To scale to the full
1000-article set, `src/evaluation/testset.py` does the same span->chunk mapping in bulk.